# 02 - Stationboard fetch with retry

Fetches the departure board of **one station** from `transport.opendata.ch/v1/stationboard`.
This function is the building block of the ingestion job, which calls it once per station.

## Error handling policy

| Situation | Behaviour |
|---|---|
| Timeout / connection error | transient: retry with exponential backoff (2 s, 4 s), max 3 attempts |
| HTTP 429, 500, 502, 503, 504 | transient: retry with exponential backoff, max 3 attempts |
| Any other non-200 status | permanent: fail immediately |
| HTTP 200 but departures belong to another station | permanent: fail immediately (see test 3) |

Every request has a timeout, so a hanging server cannot block the job.

In [1]:
import time
import requests

API_URL = "https://transport.opendata.ch/v1/stationboard"
RETRYABLE_STATUS = {429, 500, 502, 503, 504}   # transient server-side errors


def fetch_stationboard(station_id, limit=30, max_attempts=3, timeout_s=15):
    """Fetch the next departures of one station.

    Retries transient failures (network errors, HTTP 429/5xx) with exponential
    backoff and fails immediately on permanent ones.

    Returns:
        (departures, attempts_used)
    Raises:
        requests.RequestException or RuntimeError when the call finally fails.
    """
    for attempt in range(1, max_attempts + 1):
        is_last = attempt == max_attempts
        wait_s = 2 ** attempt                      # backoff: 2 s, then 4 s

        # Transient network failure: retry, or re-raise the original error on the last attempt
        try:
            response = requests.get(
                API_URL,
                params={"id": station_id, "limit": limit},
                timeout=timeout_s,
            )
        except (requests.Timeout, requests.ConnectionError) as exc:
            if is_last:
                raise
            print(f"[{station_id}] attempt {attempt} failed ({type(exc).__name__}), retrying in {wait_s}s")
            time.sleep(wait_s)
            continue

        # Transient server-side error: retry
        if response.status_code in RETRYABLE_STATUS:
            if is_last:
                raise RuntimeError(f"HTTP {response.status_code} after {attempt} attempts")
            print(f"[{station_id}] attempt {attempt} failed (HTTP {response.status_code}), retrying in {wait_s}s")
            time.sleep(wait_s)
            continue

        # Any other non-200 status is permanent: retrying would not change the result
        if response.status_code != 200:
            raise RuntimeError(f"HTTP {response.status_code} (permanent error)")

        departures = response.json()["stationboard"]

        # The API does not reject unknown ids: it falls back to a name search and still
        # returns HTTP 200 with another station's departures. Validate every departure
        # so a wrong id fails loudly instead of storing wrong data.
        # An empty board (e.g. at night) is valid: the loop simply does not run.
        for dep in departures:
            returned_id = dep["stop"]["station"]["id"]
            if returned_id != station_id:
                raise RuntimeError(f"Station mismatch: asked for {station_id}, got {returned_id}")

        return departures, attempt

## Test 1 - Nominal case
Biel/Bienne (`8504300`). Expected: a list of departures, returned in 1 attempt.

In [2]:
deps, attempts = fetch_stationboard("8504300")
print(len(deps), "departures in", attempts, "attempt(s)")

30 departures in 1 attempt(s)


## Test 2 - Network failure
Forced with a 1 ms timeout. Expected: 2 retries with backoff (2 s, 4 s), then the original exception on the 3rd attempt.

In [3]:
fetch_stationboard("8504300", timeout_s=0.001)

[8504300] attempt 1 failed (ConnectionError), retrying in 2s
[8504300] attempt 2 failed (ConnectionError), retrying in 4s


ConnectionError: HTTPSConnectionPool(host='transport.opendata.ch', port=443): Max retries exceeded with url: /v1/stationboard?id=8504300&limit=30 (Caused by NewConnectionError("HTTPSConnection(host='transport.opendata.ch', port=443): Failed to establish a new connection: [Errno 101] Network is unreachable"))

## Test 3 - Unknown station id
Expected: `RuntimeError: Station mismatch`, raised immediately without retry.

In [4]:
fetch_stationboard("abc")

RuntimeError: Station mismatch: asked for abc, got 8572885

## Test 4 - Transient server error
HTTP 503 simulated with `httpbin.org`. Expected: 2 retries with backoff, then `RuntimeError: HTTP 503 after 3 attempts`.
The `finally` block restores the real API URL even when the call fails.

In [5]:
API_URL = "https://httpbin.org/status/503"
try:
    fetch_stationboard("8504300")
finally:
    API_URL = "https://transport.opendata.ch/v1/stationboard"   # restore the real endpoint

[8504300] attempt 1 failed (HTTP 503), retrying in 2s
[8504300] attempt 2 failed (HTTP 503), retrying in 4s


RuntimeError: HTTP 503 after 3 attempts

## Observations

- **Test 1:** 30 departures for Biel/Bienne in 1 attempt.
- **Test 2:** with a 1 ms timeout, the failure surfaces as a `ConnectionError` ("Network is unreachable") rather than a `Timeout`. Both are handled by the same transient branch: 2 retries (2 s, 4 s), then the original exception is re-raised on the 3rd attempt. The 3rd attempt prints no retry message, and the re-raised exception does not state the number of attempts.
- **Test 3:** the API does not reject unknown ids. For `"abc"` it returned **HTTP 200** with the departures of *Sörenberg, Habchegg* (`8572885`), apparently a match on the station name. Without validation this would have been a **silent data error**. The station check now raises immediately: `Station mismatch: asked for abc, got 8572885`.
- **Test 4:** HTTP 503 retried twice with backoff, then `RuntimeError: HTTP 503 after 3 attempts`. The `finally` block restored the real URL.

**Data notes from the API responses**
- `stop.delay` is `null` for departures without real-time data yet (further in the future): `null` means *unknown*, not *on time*.
- `stop.platform` is `null` for buses.

**To carry over into the ingestion job**
- Include the number of attempts in every final error message (network errors currently do not).
- Log one result line per station (status, rows, attempts, error) so failures are visible in the run log.